In [3]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from pathlib import Path

project_root = Path.cwd().parent
model_dir = project_root / "contract-category-model"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

c:\Users\fulle\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 402.32it/s]


In [4]:
# define categories for classifier

relevant_categories = [
    "Cap On Liability", "Termination For Convenience", "Governing Law",
    "Anti-Assignment", "Change Of Control", "Renewal Term",
    "Exclusivity", "Non-Compete", "Notice Period To Terminate Renewal", "Liquidated Damages"
]

relevant_categories = [category.casefold() for category in relevant_categories]

In [5]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent  # if the notebook's working directory is notebooks/
sys.path.insert(0, str(repo_root))

from src.chunking import get_chunks, remove_partial

In [6]:
test_chunks = get_chunks(project_root / "data/test.json", tokenizer, relevant_categories)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2413 > 512). Running this sequence through the model will result in indexing errors


In [7]:
import torch

def evaluate(model, tokenizer, texts):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    labels = model.config.id2label  # Maps output index to category name

    batch_size = 8
    results = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start : start + batch_size]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(device)

        with torch.no_grad():
            logits = model(**inputs).logits
            probabilities = torch.sigmoid(logits)

        for scores in probabilities.cpu():
            results.append({
                labels[i]: score.item()
                for i, score in enumerate(scores)
            })
    return results

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score

# Use the same label order that was used when training.
labels = list(relevant_categories)

# Match the training setup: skip chunks with incomplete labels.
eval_chunks = remove_partial(test_chunks)

texts = eval_chunks["text"].fillna("").astype(str).tolist()



# Run your existing batched prediction loop using `texts`.
# It should create `results`, a list of dictionaries of label -> probability.
# Then apply your threshold:
results = evaluate(model, tokenizer, texts)
threshold = 0.5
predicted_categories = [
    [category for category, score in row.items() if score >= threshold]
    for row in results
]

# Convert true and predicted category lists into matching binary matrices.
true_label_lists = [
    [label for label in (categories or []) if label in labels]
    for categories in eval_chunks["categories"]
]

mlb = MultiLabelBinarizer(classes=labels)
y_true = mlb.fit_transform(true_label_lists)
y_pred = mlb.transform(predicted_categories)

print("Micro F1:", f1_score(y_true, y_pred, average="micro", zero_division=0))
print("Macro F1:", f1_score(y_true, y_pred, average="macro", zero_division=0))
print("Samples F1:", f1_score(y_true, y_pred, average="samples", zero_division=0))

Micro F1: 0.8050713153724247
Macro F1: 0.7552600104264745
Samples F1: 0.13178688524590162


The bert-base-uncased model fine tuned on the training data had a macro F1 score of 0.805